# 03b — Hyperparameter Tuning

Tunes the five required classifiers from `03_train_models.ipynb` (Decision Tree, Naive Bayes, k-NN, Random Forest, AdaBoost) plus two extra gradient-boosting candidates (XGBoost, LightGBM), using Optuna (TPE Bayesian search) scored on F1 via 5-fold CV. Tuned models are saved alongside the existing fixed-hyperparameter ones under a `_tuned` suffix so `04_evaluate_compare.ipynb` can compare both. Requires `02_preprocessing.ipynb` to have been run first.

In [1]:
import pickle

import optuna
import pandas as pd
from lightgbm import LGBMClassifier
from optuna.samplers import TPESampler
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from config import DATA_DIR, RANDOM_STATE

optuna.logging.set_verbosity(optuna.logging.WARNING)

MODELS_DIR = DATA_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

N_TRIALS = 40
CV_FOLDS = 5

## Load train splits

Same files `03_train_models.ipynb` reads: the original (imbalanced) training set and the SMOTE-balanced variant produced by `02_preprocessing.ipynb`.

In [2]:
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").iloc[:, 0]
X_train_bal = pd.read_csv(DATA_DIR / "X_train_balanced.csv")
y_train_bal = pd.read_csv(DATA_DIR / "y_train_balanced.csv").iloc[:, 0]

print(f"Original train shape: {X_train.shape}")
print(f"SMOTE-balanced train shape: {X_train_bal.shape}")

Original train shape: (32000, 25)
SMOTE-balanced train shape: (55084, 25)


## Search space definitions

One `objective(trial, X, y)` builder per model family. Each samples hyperparameters, scores 5-fold CV F1 on the given training set, and returns the mean. `scale_pos_weight` is only meaningful (and only searched) on the original imbalanced training set, not the already-balanced SMOTE one.

In [3]:
def cv_f1(model, X, y):
    scores = cross_val_score(model, X, y, scoring="f1", cv=CV_FOLDS, n_jobs=-1)
    return scores.mean()


def make_objective(name, X, y, is_balanced):
    def objective(trial):
        if name == "decision_tree":
            model = DecisionTreeClassifier(
                max_depth=trial.suggest_int("max_depth", 3, 20),
                min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 50),
                random_state=RANDOM_STATE,
            )
        elif name == "naive_bayes":
            model = GaussianNB(
                var_smoothing=trial.suggest_float("var_smoothing", 1e-12, 1e-6, log=True)
            )
        elif name == "knn":
            model = KNeighborsClassifier(
                n_neighbors=trial.suggest_int("n_neighbors", 3, 51, step=2),
                weights=trial.suggest_categorical("weights", ["uniform", "distance"]),
            )
        elif name == "random_forest":
            model = RandomForestClassifier(
                n_estimators=trial.suggest_int("n_estimators", 100, 500, step=50),
                max_depth=trial.suggest_int("max_depth", 4, 20),
                min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 20),
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
        elif name == "adaboost":
            model = AdaBoostClassifier(
                n_estimators=trial.suggest_int("n_estimators", 50, 300, step=25),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 2.0, log=True),
                random_state=RANDOM_STATE,
            )
        elif name == "xgboost":
            params = dict(
                n_estimators=trial.suggest_int("n_estimators", 100, 500, step=50),
                max_depth=trial.suggest_int("max_depth", 3, 12),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                subsample=trial.suggest_float("subsample", 0.6, 1.0),
                colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
            if not is_balanced:
                params["scale_pos_weight"] = trial.suggest_float("scale_pos_weight", 1.0, 8.0)
            model = XGBClassifier(**params)
        elif name == "lightgbm":
            params = dict(
                n_estimators=trial.suggest_int("n_estimators", 100, 500, step=50),
                max_depth=trial.suggest_int("max_depth", 3, 12),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                num_leaves=trial.suggest_int("num_leaves", 15, 127),
                subsample=trial.suggest_float("subsample", 0.6, 1.0),
                colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbosity=-1,
            )
            if not is_balanced:
                params["scale_pos_weight"] = trial.suggest_float("scale_pos_weight", 1.0, 8.0)
            model = LGBMClassifier(**params)
        else:
            raise ValueError(name)

        return cv_f1(model, X, y)

    return objective


def build_tuned_model(name, best_params):
    if name == "decision_tree":
        return DecisionTreeClassifier(**best_params, random_state=RANDOM_STATE)
    if name == "naive_bayes":
        return GaussianNB(**best_params)
    if name == "knn":
        return KNeighborsClassifier(**best_params)
    if name == "random_forest":
        return RandomForestClassifier(**best_params, random_state=RANDOM_STATE, n_jobs=-1)
    if name == "adaboost":
        return AdaBoostClassifier(**best_params, random_state=RANDOM_STATE)
    if name == "xgboost":
        return XGBClassifier(**best_params, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)
    if name == "lightgbm":
        return LGBMClassifier(**best_params, random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1)
    raise ValueError(name)

## Run tuning

For each model family, run one Optuna study against the original training set and one against the SMOTE-balanced training set (`N_TRIALS` trials each, TPE sampler seeded for reproducibility). Refit the best-params model on the full respective training set and save it to `data/models/{name}_{orig|smote}_tuned.pkl`.

In [4]:
MODEL_NAMES = ["decision_tree", "naive_bayes", "knn", "random_forest", "adaboost", "xgboost", "lightgbm"]

DATASETS = {
    "orig": (X_train, y_train, False),
    "smote": (X_train_bal, y_train_bal, True),
}

tuning_summary = []

for suffix, (X, y, is_balanced) in DATASETS.items():
    for name in MODEL_NAMES:
        study = optuna.create_study(
            direction="maximize",
            sampler=TPESampler(seed=RANDOM_STATE),
        )
        study.optimize(make_objective(name, X, y, is_balanced), n_trials=N_TRIALS, show_progress_bar=False)

        best_model = build_tuned_model(name, study.best_params)
        best_model.fit(X, y)

        out_path = MODELS_DIR / f"{name}_{suffix}_tuned.pkl"
        with open(out_path, "wb") as f:
            pickle.dump(best_model, f)

        tuning_summary.append({
            "model": name,
            "dataset": suffix,
            "best_cv_f1": study.best_value,
            "best_params": study.best_params,
        })
        print(f"[{suffix}] {name}: best CV F1={study.best_value:.4f}  params={study.best_params}")
        print(f"  saved: {out_path.name}")

[orig] decision_tree: best CV F1=0.2928  params={'max_depth': 20, 'min_samples_leaf': 1}
  saved: decision_tree_orig_tuned.pkl
[orig] naive_bayes: best CV F1=0.4193  params={'var_smoothing': 1.7670169402947963e-10}
  saved: naive_bayes_orig_tuned.pkl
[orig] knn: best CV F1=0.2825  params={'n_neighbors': 3, 'weights': 'uniform'}
  saved: knn_orig_tuned.pkl
[orig] random_forest: best CV F1=0.2005  params={'n_estimators': 500, 'max_depth': 16, 'min_samples_leaf': 1}
  saved: random_forest_orig_tuned.pkl
[orig] adaboost: best CV F1=0.2684  params={'n_estimators': 175, 'learning_rate': 1.3912260642730854}
  saved: adaboost_orig_tuned.pkl
[orig] xgboost: best CV F1=0.4665  params={'n_estimators': 250, 'max_depth': 5, 'learning_rate': 0.010044594463394242, 'subsample': 0.9549772951275722, 'colsample_bytree': 0.9933590021804523, 'scale_pos_weight': 3.4867975358464123}
  saved: xgboost_orig_tuned.pkl
[orig] lightgbm: best CV F1=0.4689  params={'n_estimators': 500, 'max_depth': 3, 'learning_rate

## Tuning summary

In [5]:
summary_df = pd.DataFrame(tuning_summary).sort_values("best_cv_f1", ascending=False)
summary_df.to_csv(DATA_DIR / "hyperparameter_tuning_summary.csv", index=False)
summary_df

,model,dataset,best_cv_f1,best_params
9,knn,smote,0.884273,"{'n_neighbors': 3, 'weights': 'distance'}"
10,random_forest,smote,0.883653,"{'n_estimators': 450, 'max_depth': 18, 'min_sa..."
12,xgboost,smote,0.858921,"{'n_estimators': 150, 'max_depth': 10, 'learni..."
13,lightgbm,smote,0.856027,"{'n_estimators': 250, 'max_depth': 4, 'learnin..."
7,decision_tree,smote,0.852276,"{'max_depth': 20, 'min_samples_leaf': 2}"
11,adaboost,smote,0.848443,"{'n_estimators': 275, 'learning_rate': 0.64669..."
8,naive_bayes,smote,0.720424,{'var_smoothing': 1.7670169402947963e-10}
6,lightgbm,orig,0.468895,"{'n_estimators': 500, 'max_depth': 3, 'learnin..."
5,xgboost,orig,0.466466,"{'n_estimators': 250, 'max_depth': 5, 'learnin..."
1,naive_bayes,orig,0.419347,{'var_smoothing': 1.7670169402947963e-10}
